# 🔎 100 Regex Ideas: Try-It Notebook
**OPIM 5512 · Module 3** · goes with the *100 Regex Ideas* sheet (HuskyCT → Lab 3 folder)

**First: File → Save a copy in Drive.**

Run the setup cells, then run any section. Every line tries one idea on **all** your ads and prints:
how many ads matched, plus a few examples of what it grabbed. Change a pattern, re-run, and see what happens. That is the fastest way to learn regex.

- `USE_BACKUP_DATA = True` uses Dr. Wanik's 40 Hartford ads (no sign-in).
- `USE_BACKUP_DATA = False` copies the ads from **your** bucket (pick your Google Cloud Gmail in the pop-up).

In [ ]:
USE_BACKUP_DATA = True               # True = Dr. Wanik's Hartford sample (no sign-in)
PROJECT_ID = "YOUR-PROJECT-ID"      # only needed when USE_BACKUP_DATA = False
BUCKET_NAME = "YOUR-PROJECT-ID"     # usually the same as your project ID

In [ ]:
import glob
import io
import os
import re
import zipfile
import urllib.request
import pandas as pd

DATA_DIR = "/content/bucket_copy"
os.makedirs(DATA_DIR, exist_ok=True)

if USE_BACKUP_DATA:
    SAMPLE_URL = "https://raw.githubusercontent.com/drdave-teaching/OPIM5512-notebooks/main/Module3/data/lab3_hartford_sample.zip"
    zip_bytes = urllib.request.urlopen(SAMPLE_URL).read()
    zipfile.ZipFile(io.BytesIO(zip_bytes)).extractall(DATA_DIR)
    print("Loaded Dr. Wanik's Hartford backup data")
else:
    from google.colab import auth
    from google.cloud import storage
    auth.authenticate_user()
    client = storage.Client(project=PROJECT_ID)
    n_copied = 0
    for blob in client.list_blobs(BUCKET_NAME):
        wanted = blob.name.endswith(".txt") or blob.name.endswith(".csv")
        if wanted and not blob.name.startswith("state/"):
            local_path = os.path.join(DATA_DIR, blob.name)
            os.makedirs(os.path.dirname(local_path), exist_ok=True)
            blob.download_to_filename(local_path)
            n_copied = n_copied + 1
    print("Copied", n_copied, "files from gs://" + BUCKET_NAME)

ads = []
for path in sorted(glob.glob(os.path.join(DATA_DIR, "scrapes", "*", "*.txt"))):
    with open(path, encoding="utf-8") as f:
        ads.append(f.read())

cars = pd.read_csv(os.path.join(DATA_DIR, "structured/datasets/listings_master.csv"), dtype={"post_id": str})
paths = list(cars["source_txt"])
print(len(ads), "ads loaded")

## The `show()` helper
You don't need to change this. It runs one pattern on every ad and prints a one-line report card.

In [ ]:
def show(name, pattern, flags=0, mode="search", repl="", demo=None, group=None, texts=None):
    if texts is None:
        texts = ads
    if mode == "split":
        print(name, "->", re.split(pattern, demo))
        return
    hits = 0
    examples = []
    for text in texts:
        if mode == "sub":
            new_text = re.sub(pattern, repl, text, flags=flags)
            if new_text != text:
                hits = hits + 1
            continue
        if mode == "findall":
            found = re.findall(pattern, text, flags)
            if found:
                hits = hits + 1
                if len(examples) < 3:
                    examples.append(found[:3])
            continue
        m = re.search(pattern, text, flags)
        if m:
            hits = hits + 1
            if group == 0 or len(m.groups()) == 0:
                value = m.group(0)
            elif len(m.groups()) == 1:
                value = m.group(1)
            else:
                value = m.groups()
            if value not in examples and len(examples) < 3:
                examples.append(value)
    word = " matched"
    if mode == "sub":
        word = " changed"
    report = name + "  |  " + str(hits) + " of " + str(len(texts)) + word
    if mode == "sub" and demo is not None:
        report = report + "  |  " + repr(demo) + " -> " + repr(re.sub(pattern, repl, demo, flags=flags))
    if examples:
        report = report + "  |  e.g. " + repr(examples)[:120]
    print(report)

## 1. The details block (label: value)
The easiest wins. Craigslist prints a tidy block of `label:` lines, and the value is always on the NEXT line. Learn this pattern and you get half the dataset for free.

In [ ]:
show("1.1 Condition", r"condition:\n(.+)")
show("1.2 Title status", r"title status:\n(\w+)")
show("1.3 Transmission", r"transmission:\n(\w+)")
show("1.4 Drive (fwd / rwd / 4wd)", r"drive:\n(\w+)")
show("1.5 Cylinders as a number", r"cylinders:\n(\d+)")
show("1.6 Body type", r"^type:\n(\w+)", re.M)
show("1.7 Paint color (only if listed)", r"paint color:\n(\w+)")
show("1.8 Only the NON-clean titles", r"title status:\n(?!clean)(\w+)")
show("1.9 Every label and value at once", r"^([a-z ]+):\n(.+)$", re.M, mode="findall")
show("1.10 The details block, cut out whole", r"^(?:19|20)\d{2}\n.*\n((?:[a-z ]+:\n.+\n)+)", re.M)

## 2. Year
Every ad has the year in 2 or 3 places. The trick is grabbing the RIGHT 4 digits.

In [ ]:
show("2.1 Any 4 digits", r"\d{4}")
show("2.2 Starts with 19 or 20", r"(19|20)\d{2}", group=0)
show("2.3 Whole numbers only", r"\b(19|20)\d{2}\b", group=0)
show("2.4 Year at the very start of the file", r"\A(\d{4})")
show("2.5 A line that is ONLY a year", r"^((?:19|20)\d{2})$", re.M)
show("2.6 Year from the title line", r"^(\d{4}) .+ for sale by owner")
show("2.7 The POSTING year (not the car year!)", r"Posted\n(\d{4})-")
show("2.8 Only plausible car years", r"\b(19[5-9]\d|20[0-2]\d)\b")
show("2.9 Year between line breaks, with lookarounds", r"(?<=\n)(?:19|20)\d{2}(?=\n)")
show("2.10 Do the years disagree?", r"\b(?:19|20)\d{2}\b", mode="findall")

## 3. Dates, times and IDs
The easy structured stuff: always the same shape, so it is great practice with exact counts.

In [ ]:
show("3.1 The post id", r"post id: (\d+)")
show("3.2 A date", r"\d{4}-\d{2}-\d{2}")
show("3.3 Year, month and day as separate groups", r"(\d{4})-(\d{2})-(\d{2})")
show("3.4 A clock time", r"\b\d{2}:\d{2}\b")
show("3.5 Date AND time after 'Posted'", r"Posted\n(\d{4}-\d{2}-\d{2} \d{2}:\d{2})")
show("3.6 Named groups", r"(?P<year>\d{4})-(?P<month>\d{2})-(?P<day>\d{2})")
show("3.7 Hour of posting (a feature!)", r"Posted\n\d{4}-\d{2}-\d{2} (\d{2}):")
show("3.8 How many photos", r"image 1 of (\d+)")
show("3.9 A VIN, anywhere", r"\b[A-HJ-NPR-Z0-9]{17}\b")
show("3.10 Run id and post id from a file path", r"scrapes/(\d{14})/([A-Za-z0-9]+)\.txt", texts=paths)

## 4. Price
The dollar sign is a special character, commas break numbers apart, and sellers write prices every way imaginable.

In [ ]:
show("4.1 A dollar sign and digits", r"\$\d+")
show("4.2 Let commas in", r"\$[\d,]+")
show("4.3 Negotiable?", r"\b(obo|or best offer|best offer)\b", re.I)
show("4.4 Firm price?", r"\b(firm|non[- ]?negotiable)\b", re.I)
show("4.5 Real thousands formatting", r"\$\d{1,3}(?:,\d{3})*")
show("4.6 The price line in the header", r"^\$([\d,]+)$", re.M)
show("4.7 The price after the title dash", r"-\n\$([\d,]+)")
show("4.8 Clean it into a number", r"\D", mode="sub", repl="", demo="$11,500")
show("4.9 Every dollar amount in the ad", r"\$\s*:?\s*(\d[\d,.]*)", mode="findall")
show("4.10 '14k OBO' style prices", r"\b(\d+(?:\.\d)?)\s?k\s*(?:obo|or best offer|firm)", re.I)

## 5. Mileage
The odometer field is clean, but half the useful mileage info lives in the free text: `130k`, `88K miles`, `41,534 ORIGINAL MILES`.

In [ ]:
show("5.1 The odometer field", r"odometer:\n(.+)")
show("5.2 Just the number", r"odometer:\n([\d,]+)")
show("5.3 '... miles' in the text", r"([\d,]+) miles", re.I)
show("5.4 The 'k' shorthand", r"\b(\d{2,3})\s?k\b", re.I)
show("5.5 'k miles'", r"\b(\d{2,3})\s?k\s+miles\b", re.I)
show("5.6 'ORIGINAL miles', even misspelled", r"([\d,]+)\s+or?i?ginal\s+miles", re.I)
show("5.7 Bragging words", r"\b(?:only|just)\s+([\d,]+(?:\s?k)?)\b", re.I)
show("5.8 Either format with alternation", r"\b(\d{1,3}(?:,\d{3})+|\d{2,3}\s?k)\s*miles", re.I)
show("5.9 Skip 'miles on the motor'", r"([\d,]+)\s+miles(?!\s+on\s+(?:the\s+)?(?:motor|engine))", re.I, mode="findall")
show("5.10 k-miles that are NOT prices", r"\b(\d{2,3})\s?k\b(?!\s*(?:obo|or best|firm))", re.I)

## 6. Paint color
Only some sellers fill in `paint color:`, so the rest of the color info hides in sentences like `BURGUNDY WITH TAN TWEED INTERIOUR`.

In [ ]:
show("6.1 The naive search", r"red", re.I)
show("6.2 Whole word only", r"\bred\b", re.I)
show("6.3 The paint color field", r"paint color:\n(\w+)")
show("6.4 gray or grey", r"\bgr[ae]y\b", re.I)
show("6.5 A list of colors", r"\b(red|blue|black|white|silver|gr[ae]y|green|burgundy|tan|gold)\b", re.I)
show("6.6 '___ exterior'", r"(\w+) exterior", re.I)
show("6.7 Shade words", r"\b(dark|light|bright|metallic)\s+(red|blue|gr[ae]y|green|silver)\b", re.I)
show("6.8 Exterior/Interior pairs", r"\b(\w+)\s*/\s*(\w+)\s+(?:\w+\s+)?interiou?r\b", re.I)
show("6.9 'COLOR WITH ... INTERIOR' lines", r"^([a-z ]+?) with .*interiou?r", re.I | re.M)
show("6.10 Field first, text as backup", r"paint color:\n(\w+)|\b(red|blue|black|white|silver|gr[ae]y|burgundy)\b", re.I)

## 7. Location
Where is the car? The title line ends with `Town, ST - craigslist`, and sellers add their own clues.

In [ ]:
show("7.1 Town from the title line", r" - (.+?), CT - craigslist")
show("7.2 The Craigslist area", r"CL\n(\w+)\n>")
show("7.3 A list of towns", r"\b(hartford|new britain|bristol|manchester|enfield|vernon)\b", re.I)
show("7.4 Town AND state", r" - ([^,\n]+), ([A-Z]{2}) - craigslist")
show("7.5 Neighborhood in parentheses", r"^\(([^)\n]+)\)$", re.M)
show("7.6 Compass towns", r"\b(East|West|South|North) ([A-Z][a-z]+)\b")
show("7.7 'Located in ...'", r"located in ([A-Z][\w ]+?)(?:[.,\n]|$)", re.I)
show("7.8 'Location : ...' lines", r"^location\s*:\s*(.+)$", re.I | re.M)
show("7.9 Split a list of towns", r"\s*/\s*", mode="split", demo="Bristol/ Plainville / New Britain CT")
show("7.10 In-state or out-of-state?", r", (?!CT\b)([A-Z]{2}) - craigslist")

## 8. What the seller says (flags for your model)
Turn free text into yes/no features: new parts, problems, owners, emissions. This is feature engineering with regex.

In [ ]:
show("8.1 New tires?", r"new tires", re.I)
show("8.2 New tires, brakes or battery", r"\bnew (tires|brakes|battery)\b", re.I)
show("8.3 Problem words", r"\b(rust|dents?|crack|damage|leaks?)\b", re.I)
show("8.4 One owner", r"\b(1|one|original) owners?\b", re.I)
show("8.5 Everything that is 'new'", r"\bnew (\w+)", re.I, mode="findall")
show("8.6 What it NEEDS", r"\bneeds? (?!nothing)(\w+(?: \w+)?)", re.I, mode="findall")
show("8.7 Emissions done", r"emissions?\s+(?:done|ready|passed)|passed\s+(?:ct\s+)?emissions", re.I)
show("8.8 Service records count", r"(\d+) service records", re.I)
show("8.9 Problems that are NOT denied", r"(?<!no )(?<!no fluid )\b(rust|dents?|crack|damage|leaks)\b(?! free)", re.I)
show("8.10 SHOUTING lines", r"^[A-Z0-9 ,.'!&/()-]{15,}$", re.M, mode="findall")

## 9. Clean-up and privacy (re.sub)
`re.sub(pattern, replacement, text)` finds AND replaces. It is the tool for tidying text before a model or an LLM sees it, and for scrubbing personal information.

In [ ]:
show("9.1 Remove commas", r",", mode="sub", repl="", demo="170,000")
show("9.2 Collapse extra spaces", r"[ \t]+", mode="sub", repl=" ", demo="ratrod  327 v8")
show("9.3 Fix a known typo", r"\bhyunda\b", re.I, mode="sub", repl="hyundai", demo="2013 Hyunda Sonata")
show("9.4 Fix a family of typos", r"\binteriou?r\b", re.I, mode="sub", repl="interior", demo="TAN LEATHER INTERIOUR")
show("9.5 Remove emojis and odd symbols", r"[^\x00-\x7F]+", mode="sub", repl="", demo="Emissions Ready✅🔌")
show("9.6 Keep only the description", r"QR Code Link to This Post\n(.*?)\npost id:", re.S)
show("9.7 Cut the header junk", r"\A.*?QR Code Link to This Post\n", re.S, mode="sub", repl="")
show("9.8 Redact phone numbers", r"(?<!id: )\(?\b\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4}\b", mode="sub", repl="[PHONE]")
show("9.9 Redact email addresses", r"[\w.+-]+@[\w-]+\.[\w.]+", mode="sub", repl="[EMAIL]")
show("9.10 Catch sneaky phone numbers", r"\d*(?:zero|one|two|three|four|five|six|seven|eight|nine)\d*[-\s]?\d{3}[-\s]?\d{2}[-\s]?\d{2}", re.I)

## 10. Make and model (the boss level)
The field the extractor gets wrong. Build up to it here, then finish it in **Lab 3, Way 2**.

In [ ]:
show("10.1 One brand", r"honda", re.I)
show("10.2 A list of brands", r"\b(honda|toyota|ford|chevy|chevrolet|bmw|jeep|acura|hyundai|nissan)\b", re.I)
show("10.3 Both spellings of Chevy", r"\bchev(y|rolet)\b", re.I, group=0)
show("10.4 Hyphen or no hyphen", r"\b(cr-?v|hr-?v|rav-?4|f-?150)\b", re.I)
show("10.5 First word after the title year", r"\A\d{4} (\w+)")
show("10.6 Make AND model from the title", r"\A\d{4} (\S+) (\S+)")
show("10.7 The whole title before the boilerplate", r"\A(.+?) for sale by owner")
show("10.8 The model after a known make", r"\b(?:honda|toyota|ford|jeep|acura)\s+([\w-]+)", re.I)
show("10.9 Why the current extractor fails", r"\b([A-Z][a-z]+)\s+([A-Z][A-Za-z0-9]+)")
# 10.10 is your Lab 3 Way 2 challenge. Fill in the two groups, then remove the # below:
# show("10.10 Lab 3 Way 2", r"^(?:19|20)\d{2}\n(______)\s+(______)", re.M)

## ✏️ Your turn
Write your own pattern for a field your price model could use, and test it here.

In [ ]:
show("my idea", r"type your pattern here", re.I)